# <font color='#E8800A'>Import and load the data</font> <a class="anchor" id="import"></a>

In [10]:
import joblib
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit, cross_validate
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)  # For reproducibility

# Saved by the pre-processing notebook (train rows are sorted by date)
train = pd.read_csv("../data/pre-processed/train_clean.csv", index_col="ID")
target = "repeat_purchase_90d"
X, y = train.drop(columns=[target]), train[target]
X_test = pd.read_csv("../data/pre-processed/test_clean.csv", index_col="ID")

# Saved by the feature selection notebook: the pipeline using only the selected features
preprocess = joblib.load("../data/pre-processed/preprocess_selected.joblib")
selected_features = joblib.load("../data/pre-processed/selected_features.joblib")

print(f"train: {len(X):,} rows | test: {len(X_test):,} rows | {len(selected_features)} selected features")

train: 6,469 rows | test: 986 rows | 22 selected features


# <font color='#E8800A'>Assessment strategy</font> <a class="anchor" id="assessment"></a>

**5-fold time-series cross-validation** (`TimeSeriesSplit`), the same folds used in pre-processing and feature selection. Each fold trains on all purchases up to a date and validates on the next block.

Why this strategy:

- **`test.csv` is entirely later than `train.csv`.** So we always validate on purchases that come *after* the ones we train on, like the real task. A random split would let the model learn from the future and give scores that look better than they really are.
- **Customers buy many times.** A random split would put purchases of the same customer, days apart, on both sides. Splitting by time mostly avoids this.
- **5 folds instead of one split.** Each score is an average over 5 periods, from 2017 to 2025, so one lucky period can't decide. The spread between folds shows how stable a model is.

All the learning (capping limits, medians, scaling, encoding, and the model itself) happens inside the pipeline, on the training rows of each fold only. `test.csv` is never used to fit anything.

In [11]:
time_split = TimeSeriesSplit(n_splits=5)

for fold, (train_rows, val_rows) in enumerate(time_split.split(X), start=1):
    dates = X["purchase_date"]
    print(f"fold {fold}: train {len(train_rows):,} rows up to {dates.iloc[train_rows].max()}"
          f" | validate {len(val_rows):,} rows to {dates.iloc[val_rows].max()}"
          f" | repeat rate in validation {y.iloc[val_rows].mean():.2f}")

fold 1: train 1,079 rows up to 2017-08-12 | validate 1,078 rows to 2019-01-12 | repeat rate in validation 0.59
fold 2: train 2,157 rows up to 2019-01-12 | validate 1,078 rows to 2020-10-03 | repeat rate in validation 0.53
fold 3: train 3,235 rows up to 2020-10-03 | validate 1,078 rows to 2022-03-05 | repeat rate in validation 0.58
fold 4: train 4,313 rows up to 2022-03-05 | validate 1,078 rows to 2023-09-25 | repeat rate in validation 0.52
fold 5: train 5,391 rows up to 2023-09-25 | validate 1,078 rows to 2025-02-15 | repeat rate in validation 0.60


# <font color='#E8800A'>Metrics</font> <a class="anchor" id="metrics"></a>

56% of the training purchases are repeats (between 52% and 60% depending on the period, see above). That makes some common metrics misleading, so before any real model we score one that **always predicts 1** ("will repeat") without looking at the data.

In [12]:
metrics = ["f1_macro", "roc_auc", "balanced_accuracy", "f1", "accuracy"]


def assess(model):
    """Mean score over the 5 time-series folds for each metric, with the pre-processing in front."""
    scores = cross_validate(Pipeline([("preprocess", preprocess), ("model", model)]),
                            X, y, cv=time_split, scoring=metrics)
    return {metric: scores[f"test_{metric}"].mean().round(3) for metric in metrics}


results = {"always predict 1": assess(DummyClassifier(strategy="constant", constant=1))}
pd.DataFrame(results).T

,f1_macro,roc_auc,balanced_accuracy,f1,accuracy
always predict 1,0.36,0.5,0.5,0.721,0.564


Always saying "repeat" already gets **F1 = 0.72**, without looking at the data. F1 only scores the "repeat" class, so it rewards predicting 1 for everyone, and accuracy has the same problem. Neither is a good main metric here.

- **Main metric: macro F1.** The F1 of each class, averaged, so "will come back" and "won't come back" count the same. Both mistakes matter to the business: missing a customer who is about to leave, and spending retention effort on one who would come back anyway. It scores the 0/1 predictions, which is what we submit. Always-1 gets only 0.36.
- **Second metric: ROC AUC.** How well the model ranks repeat buyers above the others, whatever threshold turns probabilities into 0/1. Always-1 gets 0.5, a coin flip.
- F1, accuracy and balanced accuracy are kept in the tables for comparison.

# <font color='#E8800A'>Models</font> <a class="anchor" id="models"></a>

Two models from class, each with the same pre-processing pipeline in front:

- **Logistic regression:** simple and fast, and its coefficients are easy to explain. A good reference point.
- **Random forest:** captures non-linear effects and combinations of features (for example "few purchases *and* a long gap") without any extra work.

In [13]:
results["logistic regression"] = assess(LogisticRegression(max_iter=2000))
results["random forest"] = assess(RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                                         random_state=RANDOM_STATE, n_jobs=-1))
pd.DataFrame(results).T

,f1_macro,roc_auc,balanced_accuracy,f1,accuracy
always predict 1,0.360,0.500,0.500,0.721,0.564
logistic regression,0.696,0.775,0.699,0.733,0.704
random forest,0.708,0.786,0.708,0.750,0.715


Both models clearly beat always-1 on the metrics that matter: macro F1 0.70–0.71 vs 0.36, and ROC AUC 0.78–0.79 vs 0.5. On plain F1, logistic regression is only 0.012 above always-1, which shows again why F1 alone would mislead here.

The random forest is better on every metric, so it is the one we tune.

## Tuning the random forest

A small grid search, with the same time-series folds, choosing by macro F1:

- `min_samples_leaf`: the fewest purchases allowed in a leaf. Bigger means simpler trees and less overfitting.
- `class_weight`: `"balanced"` gives the smaller class ("no repeat") more weight, so the forest doesn't lean towards predicting 1.

In [14]:
grid = GridSearchCV(
    Pipeline([("preprocess", preprocess),
              ("model", RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1))]),
    param_grid={"model__min_samples_leaf": [5, 20, 50], "model__class_weight": [None, "balanced"]},
    cv=time_split, scoring="f1_macro",
).fit(X, y)

print("best:", grid.best_params_)
(pd.DataFrame(grid.cv_results_)
 .fillna({"param_model__class_weight": "None"})
 [["param_model__min_samples_leaf", "param_model__class_weight", "mean_test_score", "std_test_score"]]
 .rename(columns={"param_model__min_samples_leaf": "min_samples_leaf", "param_model__class_weight": "class_weight",
                  "mean_test_score": "macro F1", "std_test_score": "spread between folds"})
 .sort_values("macro F1", ascending=False).round(3))

best: {'model__class_weight': 'balanced', 'model__min_samples_leaf': 20}


,min_samples_leaf,class_weight,macro F1,spread between folds
4,20,balanced,0.719,0.009
5,50,balanced,0.717,0.009
3,5,balanced,0.714,0.008
1,20,None,0.709,0.009
0,5,None,0.708,0.009
2,50,None,0.706,0.012


In [15]:
best_forest = grid.best_estimator_.named_steps["model"]
results["random forest, tuned"] = assess(best_forest)
pd.DataFrame(results).T

,f1_macro,roc_auc,balanced_accuracy,f1,accuracy
always predict 1,0.360,0.500,0.500,0.721,0.564
logistic regression,0.696,0.775,0.699,0.733,0.704
random forest,0.708,0.786,0.708,0.750,0.715
"random forest, tuned",0.719,0.790,0.722,0.744,0.722


The best setting is `min_samples_leaf=20` with `class_weight="balanced"`: macro F1 goes from 0.708 to **0.719**, and ROC AUC from 0.786 to **0.790**.

- `"balanced"` helps at every leaf size (+0.006 to +0.011). Without it, the forest predicts "repeat" too often.
- Leaf sizes 20 and 50 score almost the same (0.719 vs 0.717), so the result doesn't depend on picking an exact value.
- Plain F1 drops a little (0.750 → 0.744) because the model now predicts "repeat" less often. That's expected: F1 rewards predicting 1, and it's the trade we chose by using macro F1.

*Caveat:* the setting is chosen on the same folds it's scored on, so its scores are slightly optimistic. The grid is small (6 settings), which keeps that effect small.

# <font color='#E8800A'>Final model check</font> <a class="anchor" id="check"></a>

The tuned random forest, fold by fold, to see how stable it is.

In [16]:
final_model = Pipeline([("preprocess", preprocess), ("model", best_forest)])

per_fold = cross_validate(final_model, X, y, cv=time_split, scoring=["f1_macro", "roc_auc"])
pd.DataFrame({"macro F1": per_fold["test_f1_macro"], "ROC AUC": per_fold["test_roc_auc"]},
             index=[f"fold {fold}" for fold in range(1, 6)]).round(3)

,macro F1,ROC AUC
fold 1,0.725,0.786
fold 2,0.705,0.780
fold 3,0.727,0.802
fold 4,0.712,0.781
fold 5,0.724,0.802


What kind of mistakes it makes, on the most recent fold (the one closest to `test.csv`):

In [17]:
train_rows, val_rows = list(time_split.split(X))[-1]
final_model.fit(X.iloc[train_rows], y.iloc[train_rows])
predicted = final_model.predict(X.iloc[val_rows])

pd.crosstab(y.iloc[val_rows].values, predicted, rownames=["actual"], colnames=["predicted"])

predicted,0,1
actual,,
0,326,100
1,193,459


**Stable over time:** macro F1 stays between 0.705 and 0.727, and ROC AUC between 0.78 and 0.80, across five periods from 2017 to 2025. No period breaks down, which is a good sign for `test.csv`, an even later period.

**Mistakes on the most recent fold:**
- of the 652 purchases that were followed by a repeat, it catches 459 (70%)
- of the 426 that were not, it gets 326 right (77%)
- its most common mistake is predicting "won't come back" for a customer who did (193 purchases)

# <font color='#E8800A'>Predictions for test.csv</font> <a class="anchor" id="predictions"></a>

The final model is trained on **all** of `train.csv`; the folds were only for assessment. It then predicts every row of `test.csv`. The ID is kept unchanged, in the same order as `sample_submission.csv`.

In [18]:
final_model.fit(X, y)
predictions = final_model.predict(X_test)

submission = pd.DataFrame({target: predictions}, index=X_test.index).reset_index()

# Checks against the sample submission: same columns, same IDs in the same order, only 0/1
sample = pd.read_csv("../data/raw/sample_submission.csv")
assert list(submission.columns) == list(sample.columns)
assert submission["ID"].equals(sample["ID"])
assert submission[target].isin([0, 1]).all()

Path("../submissions").mkdir(exist_ok=True)
submission.to_csv("../submissions/Homework_Group06_submission.csv", index=False)
print(f"{len(submission)} predictions saved, {submission[target].mean():.0%} predicted to repeat")
submission.head()

986 predictions saved, 58% predicted to repeat


,ID,repeat_purchase_90d
0,10000006,1
1,10000008,1
2,10000010,1
3,10000020,0
4,10000025,0


## Summary

| | |
|---|---|
| assessment | 5-fold time-series cross-validation (always validating on later purchases) |
| main metric | macro F1, with ROC AUC as second metric |
| final model | random forest (300 trees, `min_samples_leaf=20`, `class_weight="balanced"`) on the 22 selected features |
| cross-validated score | macro F1 0.719, ROC AUC 0.790 (always predicting 1: 0.36 and 0.5) |
| submission | `submissions/Homework_Group06_submission.csv`: 986 rows, `ID` unchanged, same format as `sample_submission.csv` |